# Real Data Performance

This notebook compares the performance of the EM approach to LOOCV approach using real-world datasets.

## Preview Experiment

In [1]:
import numpy as np
import pandas as pd

from fastridge import RidgeEM, RidgeLOOCV
from experiments import run_real_data_experiments
from problems import EmpiricalDataProblem

problems = [
    EmpiricalDataProblem('abalone',    'Rings'),
    EmpiricalDataProblem('airfoil',    'scaled-sound-pressure'),
    EmpiricalDataProblem('concrete',   'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',   'target'),
    EmpiricalDataProblem('forest',     'area'),
    EmpiricalDataProblem('student',    'G3'),
    EmpiricalDataProblem('yacht',      'Residuary_resistance'),   # much lower r2 (0.65 vs. 0.97)
    EmpiricalDataProblem('automobile', 'price', nan_policy='drop_rows'),
]

estimators = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results = run_real_data_experiments(problems, estimators, n_iterations=10, seed=1, verbose=True)
print()

abalone (n=4177, p=9)..........
airfoil (n=1503, p=5)..........
concrete (n=1030, p=8)..........
diabetes (n=442, p=10)..........
forest (n=517, p=27)..........
student (n=649, p=41)..........
yacht (n=308, p=6)..........
automobile (n=159, p=51)..........



In [2]:
rows = []
for problem, data_result in zip(problems, results):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = data_result['EM']['p']
    row['n_train']  = data_result['EM']['n_train']
    row['n:p']      = data_result['EM']['n_train'] / data_result['EM']['p']
    rows.append(row)
pd.DataFrame(rows).sort_values('n_train', ascending=False).round(2)

,dataset,target,EM,CV_glm,CV_fix,Speed-Up,p,n_train,n:p
0,abalone,Rings,0.54,0.54,0.54,8.85,9.0,2923,324.78
1,airfoil,scaled-sound-pressure,0.52,0.52,0.52,9.55,5.0,1052,210.40
2,concrete,Concrete compressive strength,0.59,0.59,0.59,8.88,8.0,721,90.12
5,student,G3,0.83,0.83,0.83,5.88,41.0,454,11.07
4,forest,area,-0.07,-0.27,-0.09,2.60,26.3,361,13.73
3,diabetes,target,0.47,0.47,0.47,8.36,10.0,309,30.90
6,yacht,Residuary_resistance,0.65,0.65,0.65,9.60,6.0,215,35.83
7,automobile,price,0.89,0.89,0.89,4.98,50.4,111,2.20


## Full Experiment

In [ ]:
problems_full = [
    EmpiricalDataProblem('abalone',          'Rings'),
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    EmpiricalDataProblem('automobile',       'price',               nan_policy='drop_rows'),
    EmpiricalDataProblem('autompg',          'mpg',                 nan_policy='drop_rows'),
    EmpiricalDataProblem('crime',            'ViolentCrimesPerPop',
                         drop=['state', 'fold', 'communityname'],
                         nan_policy='drop_cols'),
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    EmpiricalDataProblem('facebook',         'Total Interactions',  nan_policy='drop_rows'),  # 'like' and 'share' are also candidate targets
    EmpiricalDataProblem('forest',           'area'),
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']),
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3'),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results_full = run_real_data_experiments(problems_full, estimators_full,
                                         n_iterations=100, seed=1, verbose=True)
print()

In [4]:
rows_full = []
for problem, data_result in zip(problems_full, results_full):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_full.append(row)
pd.DataFrame(rows_full).sort_values('n_train', ascending=False).round(2)

,dataset,target,EM,CV_glm,CV_fix,Speed Up Ratio,p,n_train,n:p
9,naval_propulsion,GT_compressor_decay,0.84,0.84,0.84,6.95,15.00,8353,556.87
10,naval_propulsion,GT_turbine_decay,0.91,0.91,0.91,7.04,15.00,8353,556.87
11,parkinsons,motor_UPDRS,0.14,0.14,0.14,5.78,19.00,4112,216.42
12,parkinsons,total_UPDRS,0.17,0.17,0.17,5.91,19.00,4112,216.42
0,abalone,Rings,0.53,0.53,0.53,8.92,9.00,2923,324.78
1,airfoil,scaled-sound-pressure,0.51,0.51,0.51,10.45,5.00,1052,210.40
5,concrete,Concrete compressive strength,0.60,0.60,0.60,9.47,8.00,721,90.12
14,student,G3,0.83,0.83,0.83,6.17,41.00,454,11.07
8,forest,area,-0.06,-0.19,-0.07,2.60,26.46,361,13.64
4,boston,medv,0.71,0.71,0.71,7.83,13.00,354,27.23
